In [ ]:
%load_ext autoreload
%autoreload 2

### spatial info in CB

Using VPNs to access the retinotopy at the input side in OL and at the output side in CB. Assign Retinotopy Index to cell type's outgoing synapses as a way to characterize the amount of retinotopy in regions of CB. Why some regions have higher RIs?
 
Slightly weaker version of this is spatial resolution in terms of RF's size or column counts. High vs low res neurons/regions. 

compute rf size for all CB types, and average rf size of inputs (only earlier hitting time).

In [ ]:
"""
This cell does the initial project setup.
If you start a new script or notebook, make sure to copy & paste this part.

A script with this code uses the location of the `.env` file as the anchor for
the whole project (= PROJECT_ROOT). Afterwards, code inside the `src` directory
are available for import.
"""
from pathlib import Path
import sys
from dotenv import load_dotenv, find_dotenv
load_dotenv()
PROJECT_ROOT = Path(find_dotenv()).parent
sys.path.append(str(PROJECT_ROOT.joinpath('src')))
print(f"Project root directory: {PROJECT_ROOT}")


In [ ]:
from utils import olc_client
c = olc_client.connect(verbose=True)

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import pickle
import scipy

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

import matplotlib as mpl
import matplotlib.pyplot as plt

import neuprint
from neuprint import fetch_neurons, fetch_adjacencies, NeuronCriteria as NC, SynapseCriteria as SC

import navis
import navis.interfaces.neuprint as neu

import connectome_interpreter as ci

# from fafbseg import flywire
# import flybrains

In [ ]:
from quan_propagation.func import count_col, alpha_shape_2d

# # Test if count_col is imported
# print(count_col)

In [ ]:
np.random.seed(3)
N = 5
theta = np.linspace(0, 1.8 * np.pi, N)
r1 = 2 + 0.3 * np.random.randn(N)
r2 = 1.5 + 0.3 * np.random.randn(N)

points = []
for i in range(N):
    points.append([r1[i] * np.cos(theta[i]+1 ), r1[i] * np.sin(theta[i]+1)])
    points.append([r2[i] * np.cos(theta[i]), r2[i] * np.sin(theta[i])])

points = np.array(points)

# Compute alpha shape with different alpha values
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
alphas = [0, 0.7, 1.07]

for ax, alpha in zip(axes, alphas):
    all_boundaries, edges = alpha_shape_2d(points, alpha)
    
    # Plot points
    ax.scatter(points[:, 0], points[:, 1], c='blue', s=20, alpha=0.6)
    # label point with their index
    for i, (x, y) in enumerate(points):
        ax.text(x, y, str(i), fontsize=12, ha='right', va='bottom')
    
    # Plot alpha shape boundary
    if all_boundaries:
        for boundary in all_boundaries:
            # connection end to start
            boundary = np.vstack([boundary, boundary[0]])
            ax.plot(boundary[:, 0], boundary[:, 1], 'r-', linewidth=2)
    ax.set_aspect('equal')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def compute_circumradius(triangle_points):
    """
    Compute the circumradius of a triangle.
    
    Parameters:
    -----------
    triangle_points : array-like, shape (3, 2)
        The three vertices of the triangle
        
    Returns:
    --------
    radius : float
        The circumradius of the triangle
    """
    pts = np.array(triangle_points)
    a, b, c = pts[0], pts[1], pts[2]
    
    # Calculate side lengths
    a_len = np.linalg.norm(b - c)
    b_len = np.linalg.norm(a - c)
    c_len = np.linalg.norm(a - b)
    
    # Calculate area using cross product
    area = 0.5 * abs(np.cross(b - a, c - a))
    
    if area == 0:
        return float('inf')
    
    # Circumradius formula: R = (abc) / (4 * Area)
    circumradius = (a_len * b_len * c_len) / (4.0 * area)
    
    return circumradius


In [ ]:
from scipy.spatial import Delaunay
from collections import defaultdict

In [ ]:
alpha = 0.8
tri = Delaunay(points)

# Find boundary edges (edges that appear in only one triangle)
edge_count = defaultdict(int)
for simplex in tri.simplices:
    pts = points[simplex]
    circumradius = compute_circumradius(pts)
    
    if alpha == 0 or circumradius < 1.0 / alpha:
        for i in range(3):
            edge = tuple(sorted([simplex[i], simplex[(i + 1) % 3]]))
            edge_count[edge] += 1

# Keep only boundary edges (those that appear once)
boundary_edges = {edge for edge, count in edge_count.items() if count == 1}

# # Order the boundary points
# if boundary_edges:
#     edge_points = order_boundary_points(boundary_edges, points)
# else:
#     edge_points = []


In [ ]:
edges = boundary_edges
edges
# edge_points

In [ ]:
from utils.config import CACHE_DIR, DATA_DIR, FIG_DIR
DATA_DIR.mkdir(parents=True, exist_ok=True)

result_dir = FIG_DIR / 'quan_propagation'
result_dir.mkdir(parents=True, exist_ok=True)

cache_dir = CACHE_DIR / 'quan_propagation'
cache_dir.mkdir(parents=True, exist_ok=True)

# End